In [113]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure, figaspect
from scipy import signal
import librosa
import librosa.display
import IPython.display as ipd

%matplotlib inline

In [114]:
def encode_symbol(bits, bits_per_symbol):
    assert (len(bits) == bits_per_symbol), "Length of bit array must match bits per symbol."
    assert (bits_per_symbol % 2 == 0), "Bits per symbol must be even."
    
    def inverse_gray(n):
        """
        Converts a Gray coded integer to its binary equivalent.
        """
        mask = n
        while mask > 0:
            mask >>= 1
            n ^= mask
        return n
    
    # Upper bits represent in-phase amplitude
    # Lower bits represent quadrature amplitude
    I_bits = bits[: bits_per_symbol//2]
    Q_bits = bits[bits_per_symbol//2 : ]
    
    # Convert bit array to integer
    I_gray, Q_gray = 0, 0
    for i in range(bits_per_symbol // 2):
        I_gray = I_gray << 1 | I_bits[i]
        Q_gray = Q_gray << 1 | Q_bits[i]
    
    # If you look at a constellation diagram for QAM, you can think of the
    # symbols as a grid we can index into. The value represented by the
    # symbol is its index gray-coded. So if we want to represent a value,
    # we take its inverse gray code to figure out which symbol it should map to
    I_idx = inverse_gray(I_gray)
    Q_idx = inverse_gray(Q_gray)
    
    # Map indices to amplitudes
    L = 2 ** (bits_per_symbol//2)
    I = I_idx * 2 - (L - 1)
    Q = Q_idx * 2 - (L - 1)
    
    # TODO: Scaling / Normalize
    
    return I + 1j * Q

assert encode_symbol([0] * 6, 6) == (-7 - 7j)
assert encode_symbol([1,0,0] * 2, 6) == (7 + 7j)
assert encode_symbol([0,0,1,1,0,1], 6) == (-5 + 5j)
assert encode_symbol([0,1,1,1], 4) == (-1 + 1j)

In [115]:
def decode_symbol(symbol, bits_per_symbol):
    assert (bits_per_symbol % 2 == 0), "Bits per symbol must be even."
    def to_gray(n):
        return n ^ (n >> 1)
    
    # Get in-phase and quadrature component from complex symbol
    I = int(np.real(symbol))
    Q = int(np.imag(symbol))
    
    # Convert I,Q samples to indices in constellation grid (symbol)
    L = 2 ** (bits_per_symbol//2)
    I_idx = (I + (L - 1)) // 2
    Q_idx = (Q + (L - 1)) // 2
    
    # Gray code of the indicies is the value encoded in the symbol
    I_val = to_gray(I_idx)
    Q_val = to_gray(Q_idx)
    
    # Combine upper and lower half into single integer
    k = (bits_per_symbol // 2)
    value = (I_val << k) + Q_val
    
    # Convert int to zero padded bit array
    bitarray = [int(bit) for bit in bin(value)[2:].zfill(bits_per_symbol)]
    
    return bitarray

print(decode_symbol(-5 + 5j, 6))
print(encode_symbol([0,0,1,1,0,1], 6))

[0, 0, 1, 1, 0, 1]
(-5+5j)


In [116]:
def encode_bytestring(bytestring, bits_per_symbol):
    # TODO: finish
    return ...